# 6 - Gold

Construção da camada Gold a partir da Silver, deixando os dados organizados e prontos para responder às perguntas de negócio do projeto.


In [0]:
from pyspark.sql import functions as F
CATALOG="main"; SCHEMA="students_performance"
SILVER_TABLE=f"{CATALOG}.{SCHEMA}.students_performance_silver"
GOLD_TABLE=f"{CATALOG}.{SCHEMA}.students_performance_gold"
df_gold=(spark.table(SILVER_TABLE).filter(F.col("dq_valid_record")).select("gender","race_ethnicity","parental_level_of_education","lunch","test_preparation_course","math_score","reading_score","writing_score","average_score"))
df_gold.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(GOLD_TABLE)
display(df_gold.limit(10))


gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score,average_score
female,group B,bachelor's degree,standard,none,72,72,74,72.66666666666667
female,group C,some college,standard,completed,69,90,88,82.33333333333333
female,group B,high school,free/reduced,none,38,60,50,49.333333333333336
male,group E,some college,standard,none,97,87,82,88.66666666666667
male,group B,some college,free/reduced,completed,59,65,66,63.333333333333336
male,group D,high school,standard,none,88,78,75,80.33333333333333
female,group D,some high school,standard,completed,61,74,72,69.0
male,group C,associate's degree,free/reduced,completed,43,45,50,46.0
male,group D,bachelor's degree,standard,completed,68,74,74,72.0
female,group B,associate's degree,free/reduced,none,52,76,70,66.0


In [0]:
dims={
"gold_performance_by_gender":["gender"],
"gold_performance_by_lunch":["lunch"],
"gold_performance_by_preparation":["test_preparation_course"],
"gold_performance_by_parent_education":["parental_level_of_education"]}
for view,cols in dims.items():
    group=", ".join(cols)
    spark.sql(f"""CREATE OR REPLACE VIEW {CATALOG}.{SCHEMA}.{view} AS SELECT {group}, COUNT(*) students, ROUND(AVG(math_score),2) avg_math_score, ROUND(AVG(reading_score),2) avg_reading_score, ROUND(AVG(writing_score),2) avg_writing_score, ROUND(AVG(average_score),2) avg_score FROM {GOLD_TABLE} GROUP BY {group}""")
spark.sql(f"""CREATE OR REPLACE VIEW {CATALOG}.{SCHEMA}.gold_performance_by_combination AS SELECT test_preparation_course,lunch,parental_level_of_education,COUNT(*) students,ROUND(AVG(average_score),2) avg_score FROM {GOLD_TABLE} GROUP BY test_preparation_course,lunch,parental_level_of_education HAVING COUNT(*) >= 10""")
print("Tabelas virtuais Gold criadas.")


Tabelas virtuais Gold criadas.
